In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

KEY_COLS = {"customers": "customer_id", "accounts": "account_id", "transactions": "transaction_id"}

def upsert_batch(micro_batch_df, batch_id, target_table, key_col):
    # if the same key appears twice in one micro-batch, keep only the newest
    w = Window.partitionBy(key_col).orderBy(F.col("updated_at").desc())
    dedup_df = (micro_batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))

    if not spark.catalog.tableExists(target_table):
        dedup_df.write.format("delta").saveAsTable(target_table)
        return

    (DeltaTable.forName(spark, target_table).alias("t")
        .merge(dedup_df.alias("s"), f"t.{key_col} = s.{key_col}")
        .whenMatchedUpdateAll(condition="s.updated_at >= t.updated_at")
        .whenNotMatchedInsertAll()
        .execute())

for tbl, fmt in [("customers", "parquet"), ("accounts", "parquet"), ("transactions", "json")]:
    key_col = KEY_COLS[tbl]
    query = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", fmt)
        .option("cloudFiles.schemaLocation", f"/Volumes/main/default/landing_zone/_schema/{tbl}")
        .load(f"/Volumes/main/default/landing_zone/{tbl}")
        .writeStream
        .foreachBatch(lambda df, bid, tbl=tbl, key_col=key_col:
                      upsert_batch(df, bid, f"main.bronze.raw_{tbl}", key_col))
        .option("checkpointLocation", f"/Volumes/main/default/landing_zone/_checkpoint/{tbl}")
        .trigger(availableNow=True)
        .start())
    query.awaitTermination()  # block until this table's batch is fully merged before moving to the next

In [0]:
%sql
CREATE OR REPLACE TABLE main.bronze.raw_commission_rules AS
SELECT * FROM parquet.`/Volumes/main/default/landing_zone/commission_rules`;

CREATE OR REPLACE TABLE main.bronze.raw_merchants AS
SELECT * FROM csv.`/Volumes/main/default/landing_zone/merchants` WITH (header = true);

CREATE OR REPLACE TABLE main.bronze.raw_billers AS
SELECT * FROM csv.`/Volumes/main/default/landing_zone/billers` WITH (header = true);

In [0]:
%sql
SELECT count(*) FROM main.bronze.raw_transactions;
SELECT count(*) FROM main.bronze.raw_customers;

In [0]:
%sql
use catalog `main`; select * from `bronze`.`raw_billers` limit 100;